## CWS for EVRP

In [8]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
import math
import os
import time

def dist_euclidean(x1, y1, x2, y2):
    return math.hypot(x1 - x2, y1 - y2)

def dist(i, j, coords):
    x1, y1 = coords[i]
    x2, y2 = coords[j]
    return dist_euclidean(x1, y1, x2, y2)

def route_distance(route, coords):
    if len(route) < 2:
        return 0
    return sum(
        dist(route[k], route[k + 1], coords)
        for k in range(len(route) - 1)
    )

def parse_instance_file(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    instance_data = {
        'name': os.path.basename(xml_path).replace('.xml', ''),
        'coords': {},
        'types': {},
        'demands': {},
        'vehicles': 1,
        'stations': 0,
        'battery_capacity': 0,
        'dimension': 0
    }

    info_elem = root.find('.//info')

    if info_elem is not None:
        name_elem = info_elem.find('.//name')

        if name_elem is not None and name_elem.text:
            instance_data['name'] = name_elem.text.strip()

    nodes_elem = root.find('.//nodes')

    if nodes_elem is not None:
        for idx, node in enumerate(nodes_elem.findall('node')):
            cx_elem = node.find('cx')
            cy_elem = node.find('cy')

            if cx_elem is not None and cy_elem is not None:
                x = float(cx_elem.text)
                y = float(cy_elem.text)

                instance_data['coords'][idx] = (x, y)

                has_tech = node.find('.//has_tech')

                if has_tech is not None and has_tech.text:
                    if has_tech.text == '1':
                        instance_data['types'][idx] = 'f'
                    else:
                        instance_data['types'][idx] = 'c'
                else:
                    instance_data['types'][idx] = 'c'

    fleet_elem = root.find('.//fleet')

    if fleet_elem is not None:
        vehicle = fleet_elem.find('.//vehicle_profile')

        if vehicle is not None:
            battery_elem = vehicle.find('.//battery_capacity')

            if battery_elem is not None and battery_elem.text:
                instance_data['battery_capacity'] = float(
                    battery_elem.text
                )

    if 0 in instance_data['coords']:
        instance_data['types'][0] = 'd'

    instance_data['stations'] = sum(
        1 for t in instance_data['types'].values() if t == 'f'
    )

    instance_data['dimension'] = len(instance_data['coords'])

    return instance_data

def parse_solution_file(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    instance_name = root.get('instance', '')

    if '/' in instance_name:
        instance_name = instance_name.split('/')[-1]

    total_cost = None

    for route in root.findall('.//route'):
        cost_elem = route.find('cost')

        if cost_elem is not None:
            total_cost = float(cost_elem.get('id', 0))
            break

    return {
        'instance': instance_name,
        'optimal_cost': total_cost
    }

def solve_cws(coords, types, depot, num_vehicles):
    customers = [
        i for i in coords
        if types.get(i) == 'c'
    ]

    if not customers:
        return [], 0

    n = len(customers)
    cust_list = list(customers)

    savings = []

    for i in range(n):
        for j in range(i + 1, n):
            ci = cust_list[i]
            cj = cust_list[j]

            saving = (
                dist(depot, ci, coords)
                + dist(depot, cj, coords)
                - dist(ci, cj, coords)
            )

            savings.append((saving, ci, cj))

    savings.sort(reverse=True, key=lambda x: x[0])

    routes = [[depot, c, depot] for c in customers]

    route_of = {
        c: i
        for i, route in enumerate(routes)
        for c in route
        if c != depot
    }

    route_first = {
        i: routes[i][1]
        for i in range(len(routes))
    }

    route_last = {
        i: routes[i][-2]
        for i in range(len(routes))
    }

    for saving, ci, cj in savings:
        if saving <= 0:
            break

        ri = route_of.get(ci)
        rj = route_of.get(cj)

        if ri is None or rj is None or ri == rj:
            continue

        if routes[ri] is None or routes[rj] is None:
            continue

        ci_is_end = (
            ci == route_first[ri]
            or ci == route_last[ri]
        )

        cj_is_end = (
            cj == route_first[rj]
            or cj == route_last[rj]
        )

        if not (ci_is_end and cj_is_end):
            continue

        route_i = routes[ri]
        route_j = routes[rj]

        route_i_clean = route_i[1:-1]
        route_j_clean = route_j[1:-1]

        best_clean = None
        best_len = float('inf')

        if ci == route_last[ri] and cj == route_first[rj]:
            new_clean = route_i_clean + route_j_clean
            new_route = [depot] + new_clean + [depot]

            d = route_distance(new_route, coords)

            if d < best_len:
                best_len = d
                best_clean = new_clean

        if ci == route_last[ri] and cj == route_last[rj]:
            new_clean = (
                route_i_clean
                + route_j_clean[::-1]
            )

            new_route = [depot] + new_clean + [depot]

            d = route_distance(new_route, coords)

            if d < best_len:
                best_len = d
                best_clean = new_clean

        if ci == route_first[ri] and cj == route_first[rj]:
            new_clean = (
                route_i_clean[::-1]
                + route_j_clean
            )

            new_route = [depot] + new_clean + [depot]

            d = route_distance(new_route, coords)

            if d < best_len:
                best_len = d
                best_clean = new_clean

        if ci == route_first[ri] and cj == route_last[rj]:
            new_clean = (
                route_j_clean
                + route_i_clean
            )

            new_route = [depot] + new_clean + [depot]

            d = route_distance(new_route, coords)

            if d < best_len:
                best_len = d
                best_clean = new_clean

        if best_clean:
            new_route = [depot] + best_clean + [depot]
            new_idx = len(routes)

            routes.append(new_route)

            for c in best_clean:
                route_of[c] = new_idx

            route_first[new_idx] = best_clean[0]
            route_last[new_idx] = best_clean[-1]

            routes[ri] = None
            routes[rj] = None

    routes = [r for r in routes if r is not None]

    while len(routes) > num_vehicles and len(routes) > 1:
        best_merge = None
        best_saving = -float('inf')

        for i in range(len(routes)):
            if routes[i] is None:
                continue

            for j in range(i + 1, len(routes)):
                if routes[j] is None:
                    continue

                saving = dist(
                    routes[i][-2],
                    routes[j][1],
                    coords
                )

                if saving > best_saving:
                    best_saving = saving

                    best_merge = (
                        i,
                        j,
                        routes[i][:-1] + routes[j][1:]
                    )

        if best_merge:
            i, j, new_route = best_merge
            routes[i] = new_route
            routes.pop(j)
        else:
            break

    total = sum(
        route_distance(r, coords)
        for r in routes
        if r is not None
    )

    return routes, total

def process_dataset_a(instances_folder, solutions_folder):
    print("=" * 80)
    print("PROCESSING DATASET A")
    print("=" * 80)

    optimal_values = {}

    if os.path.exists(solutions_folder):
        for file in os.listdir(solutions_folder):
            if file.endswith('.xml'):
                xml_path = os.path.join(
                    solutions_folder,
                    file
                )

                try:
                    sol_data = parse_solution_file(xml_path)

                    if sol_data['optimal_cost'] is not None:
                        optimal_values[
                            sol_data['instance']
                        ] = sol_data['optimal_cost']

                except:
                    pass

    results = []

    for file in os.listdir(instances_folder):
        if not file.endswith('.xml'):
            continue

        xml_path = os.path.join(instances_folder, file)

        try:
            start_time = time.time()

            instance_data = parse_instance_file(xml_path)

            instance_name = instance_data['name']
            coords = instance_data['coords']
            types = instance_data['types']

            clean_name = instance_name

            if '/nGVRP-' in clean_name:
                clean_name = clean_name.replace(
                    'evrp-data/nGVRP-',
                    ''
                )

            depot = 0

            for i, t in types.items():
                if t == 'd':
                    depot = i
                    break

            if len(coords) == 0:
                continue

            num_customers = sum(
                1 for t in types.values()
                if t == 'c'
            )

            num_stations = sum(
                1 for t in types.values()
                if t == 'f'
            )

            num_vehicles = 1

            routes, cws_distance = solve_cws(
                coords,
                types,
                depot,
                num_vehicles
            )

            exec_time = time.time() - start_time

            optimal = optimal_values.get(clean_name)

            gap = None

            if (
                optimal
                and optimal > 0
                and cws_distance > 0
            ):
                gap = (
                    (cws_distance - optimal)
                    / optimal
                ) * 100

            if clean_name.startswith('C'):
                inst_class = 'C'
            elif clean_name.startswith('R'):
                inst_class = 'R'
            elif clean_name.startswith('RC'):
                inst_class = 'RC'
            else:
                inst_class = '?'

            results.append({
                'instance': clean_name,
                'class': inst_class,
                'customers': num_customers,
                'stations': num_stations,
                'cws_km': round(cws_distance, 2),
                'optimal_km': (
                    round(optimal, 2)
                    if optimal else None
                ),
                'gap_pct': (
                    round(gap, 1)
                    if gap is not None else None
                ),
                'time_sec': round(exec_time, 4)
            })

            if optimal:
                print(
                    f"  ✅ {clean_name}: "
                    f"CWS={cws_distance:.2f}, "
                    f"Optimal={optimal:.2f}, "
                    f"Gap={gap:+.1f}%"
                )
            else:
                print(
                    f"  ✅ {clean_name}: "
                    f"CWS={cws_distance:.2f}, "
                    f"Optimal=N/A"
                )

        except Exception as e:
            print(f"  ❌ Error parsing {file}: {e}")

    return pd.DataFrame(results)

In [ ]:


if __name__ == "__main__":
    INSTANCES_FOLDER = "./EVRPMT/Dataset_A/Instances/"
    SOLUTIONS_FOLDER = "./EVRPMT/Dataset_A/Solutions/"

    if not os.path.exists(INSTANCES_FOLDER):
        print(
            f"Instances folder not found: "
            f"{INSTANCES_FOLDER}"
        )

        print("Please update the path")

    else:
        df = process_dataset_a(
            INSTANCES_FOLDER,
            SOLUTIONS_FOLDER
        )

        print("\n" + "=" * 80)
        print("SUMMARY TABLE - DATASET A")
        print("=" * 80)

        print(df.to_string(index=False))

        df.to_csv('dataset_a_results.csv', index=False)

        print(
            "\n💾 Results saved to "
            "dataset_a_results.csv"
        )

        valid = df[df['gap_pct'].notna()]

        if len(valid) > 0:
            print("\n" + "=" * 80)
            print("STATISTICS")
            print("=" * 80)

            print(f"  Total instances: {len(df)}")
            print(
                f"  Instances with optimal: "
                f"{len(valid)}"
            )

            print(
                f"  Average gap: "
                f"{valid['gap_pct'].mean():+.1f}%"
            )

            print(
                f"  Best gap: "
                f"{valid['gap_pct'].min():+.1f}%"
            )

            print(
                f"  Worst gap: "
                f"{valid['gap_pct'].max():+.1f}%"
            )

            print(
                f"  CWS better than optimal: "
                f"{len(valid[valid['gap_pct'] < 0])}"
            )

            print(
                f"  CWS worse than optimal: "
                f"{len(valid[valid['gap_pct'] > 0])}"
            )

            print("\n" + "-" * 40)
            print("BY INSTANCE CLASS")
            print("-" * 40)

            for c in ['C', 'R', 'RC']:
                subset = valid[valid['class'] == c]

                if len(subset) > 0:
                    print(
                        f"  Class {c}: "
                        f"n={len(subset)}, "
                        f"avg gap="
                        f"{subset['gap_pct'].mean():+.1f}%"
                    )

PROCESSING DATASET A
  ✅ R201-10: CWS=173.41, Optimal=189.00, Gap=-8.2%
  ✅ C202-10: CWS=231.80, Optimal=234.00, Gap=-0.9%
  ✅ C101-10: CWS=256.13, Optimal=159.00, Gap=+61.1%
  ✅ RC204-15: CWS=291.13, Optimal=295.00, Gap=-1.3%
  ✅ C103-15: CWS=244.19, Optimal=151.00, Gap=+61.7%
  ✅ R102-10: CWS=192.09, Optimal=115.00, Gap=+67.0%
  ✅ RC108-5: CWS=207.52, Optimal=108.00, Gap=+92.2%
  ✅ R102-15: CWS=238.43, Optimal=114.00, Gap=+109.2%
  ✅ R203-10: CWS=229.27, Optimal=252.00, Gap=-9.0%
  ✅ C202-15: CWS=298.94, Optimal=N/A
  ✅ RC202-15: CWS=285.49, Optimal=305.00, Gap=-6.4%
  ✅ RC208-5: CWS=168.26, Optimal=172.00, Gap=-2.2%
  ✅ C103-5: CWS=146.53, Optimal=157.00, Gap=-6.7%
  ✅ C101-5: CWS=166.80, Optimal=214.00, Gap=-22.1%
  ✅ R209-15: CWS=233.78, Optimal=264.00, Gap=-11.4%
  ✅ RC103-15: CWS=304.01, Optimal=119.00, Gap=+155.5%
  ✅ C208-15: CWS=235.97, Optimal=269.00, Gap=-12.3%
  ✅ C206-5: CWS=211.26, Optimal=205.00, Gap=+3.1%
  ✅ R103-10: CWS=143.86, Optimal=111.00, Gap=+29.6%
  ✅ C208-5: 